In [0]:
%run "/Workspace/Users/sufianaslam127@gmail.com/audit_helper"

In [0]:
from datetime import datetime, timezone

run_start = datetime.now(timezone.utc)

print("Payments Silver audit run started.")

Payments Silver audit run started.


In [0]:
from pyspark.sql.functions import (
    col,
    to_timestamp,
    lit,
    when,
    lower,
    trim,
    unix_timestamp
)
from pyspark.sql.types import DoubleType


[audit] logged run cb9c6dd1-e926-422b-98d4-ea4278ca7505 (success), latency=0.0s


In [0]:
bronze_path = "abfss://bronze@adlsnexpulse01.dfs.core.windows.net/payments/"
silver_path = "abfss://silver@adlsnexpulse01.dfs.core.windows.net/payments/"
quarantine_path = "abfss://silver@adlsnexpulse01.dfs.core.windows.net/_quarantine/payments/"

In [0]:
bronze_df = spark.read.format("delta").load(bronze_path)

bronze_count = bronze_df.count()

print(f"Bronze Payments: {bronze_count}")


Bronze Payments: 60


In [0]:
typed_df = (
    bronze_df
    .withColumn("amount", col("amount").cast(DoubleType()))
    .withColumn("event_timestamp", to_timestamp(col("timestamp")))
)

In [0]:
normalized_status = lower(trim(col("status")))

typed_df = typed_df.withColumn(
    "status",
    normalized_status
)

In [0]:
# ============================================================
# 4. ACCEPTED PAYMENT STATUSES
# ============================================================

ACCEPTED_STATUSES = [
    "successful",
    "failed",
    "pending",
    "refunded"
]

In [0]:
# ============================================================
# 5. ROW-LEVEL VALIDATION
# ============================================================

validated_df = typed_df.withColumn(
    "failure_reason",
    when(
        col("event_id").isNull(),
        lit("missing_event_id")
    )
    .when(
        col("order_id").isNull(),
        lit("missing_order_id")
    )
    .when(
        col("payment_id").isNull(),
        lit("missing_payment_id")
    )
    .when(
        col("amount").isNull() | (col("amount") < 0),
        lit("invalid_amount")
    )
    .when(
        col("event_timestamp").isNull(),
        lit("unparseable_timestamp")
    )
    .when(
        ~col("status").isin(ACCEPTED_STATUSES),
        lit("unknown_status")
    )
    .otherwise(lit(None))
)

In [0]:
# ============================================================
# 6. INGESTION DELAY / LATE EVENT FLAG
# ============================================================

validated_df = validated_df.withColumn(
    "ingestion_delay_seconds",
    unix_timestamp(col("bronze_loaded_at"))
    - unix_timestamp(col("event_timestamp"))
)

LATE_THRESHOLD_SECONDS = 120

validated_df = validated_df.withColumn(
    "is_late",
    col("ingestion_delay_seconds") > LATE_THRESHOLD_SECONDS
)

In [0]:
# ============================================================
# 7. SPLIT VALID VS QUARANTINE
# ============================================================

pre_dedup_valid_df = validated_df.filter(
    col("failure_reason").isNull()
)

quarantine_df = validated_df.filter(
    col("failure_reason").isNotNull()
)

pre_dedup_valid_count = pre_dedup_valid_df.count()

quarantined_count = quarantine_df.count()

print(f"Valid before deduplication: {pre_dedup_valid_count}")
print(f"Quarantined rows: {quarantined_count}")

Valid before deduplication: 60
Quarantined rows: 0


In [0]:
# ============================================================
# 8. DEDUPLICATION
# ============================================================

final_valid_df = pre_dedup_valid_df.dropDuplicates(
    ["event_id"]
)

final_valid_count = final_valid_df.count()

duplicates_removed_count = (
    pre_dedup_valid_count - final_valid_count
)

print(f"Final valid rows: {final_valid_count}")
print(f"Duplicates removed: {duplicates_removed_count}")

Final valid rows: 59
Duplicates removed: 1


In [0]:
# ============================================================
# 9. WRITE VALID PAYMENTS TO SILVER
# ============================================================

(
    final_valid_df.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .save(silver_path)
)

print(f"Silver Payments written: {final_valid_count}")

Silver Payments written: 59


In [0]:
# ============================================================
# 10. WRITE QUARANTINED PAYMENTS
# ============================================================

(
    quarantine_df.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .save(quarantine_path)
)

print(f"Payments quarantined: {quarantined_count}")

Payments quarantined: 0


In [0]:
# ============================================================
# 11. VERIFY SILVER PAYMENTS
# ============================================================

silver_payments_df = (
    spark.read
    .format("delta")
    .load(silver_path)
)

silver_count = silver_payments_df.count()

print(f"Silver Payments: {silver_count}")

display(
    silver_payments_df.limit(10)
)

Silver Payments: 236


event_id,payment_id,order_id,amount,payment_method,status,timestamp,kafka_ingest_time,kafka_partition,kafka_offset,bronze_loaded_at,event_timestamp,failure_reason,ingestion_delay_seconds,is_late
evt_00c8f35669,PAY_36629,ORD_25739,9446.85,cod,successful,2026-08-21T12:25:55Z,2026-08-21T12:25:58.539Z,2,10,2026-08-21T12:26:00.011Z,2026-08-21T12:25:55Z,null,5,false
evt_01879255f8,PAY_34432,ORD_7923,660.01,bank_transfer,successful,2026-08-21T12:25:48Z,2026-08-21T12:25:51.532Z,1,9,2026-08-21T12:26:00.011Z,2026-08-21T12:25:48Z,null,12,false
evt_0271d321c0,PAY_23118,ORD_44794,14194.97,wallet,successful,2026-08-21T09:15:50Z,2026-08-21T09:15:54.778Z,1,0,2026-08-21T09:16:00.018Z,2026-08-21T09:15:50Z,null,10,false
evt_033e9e9262,PAY_16168,ORD_9278,1973.39,card,refunded,2026-08-21T09:16:26Z,2026-08-21T09:16:30.787Z,1,6,2026-08-21T09:17:00.019Z,2026-08-21T09:16:26Z,null,34,false
evt_04e21a8042,PAY_72389,ORD_6568,6524.83,cod,successful,2026-08-21T12:09:50Z,2026-08-21T12:09:54.1Z,0,10,2026-08-21T12:10:00.015Z,2026-08-21T12:09:50Z,null,10,false
evt_09a8faff5e,PAY_12146,ORD_16386,14565.8,wallet,successful,2026-08-21T12:10:35Z,2026-08-21T12:10:39.129Z,2,6,2026-08-21T12:11:00.019Z,2026-08-21T12:10:35Z,null,25,false
evt_0f74a54ada,PAY_62881,ORD_41170,826.43,card,refunded,2026-08-21T09:16:15Z,2026-08-21T09:16:19.785Z,1,5,2026-08-21T09:16:30.02Z,2026-08-21T09:16:15Z,null,15,false
evt_1093affa1c,PAY_71744,ORD_32309,7751.49,bank_transfer,successful,2026-08-21T12:10:41Z,2026-08-21T12:10:45.14Z,0,21,2026-08-21T12:11:00.019Z,2026-08-21T12:10:41Z,null,19,false
evt_11d872f935,PAY_27580,ORD_36124,2261.94,wallet,failed,2026-08-21T12:10:27Z,2026-08-21T12:10:31.121Z,0,18,2026-08-21T12:11:00.019Z,2026-08-21T12:10:27Z,null,33,false
evt_14f228a957,PAY_33808,ORD_25663,7844.91,wallet,successful,2026-08-21T12:10:21Z,2026-08-21T12:10:25.12Z,0,16,2026-08-21T12:10:30.015Z,2026-08-21T12:10:21Z,null,9,false


In [0]:
# ============================================================
# 12. VERIFY SILVER SCHEMA
# ============================================================

silver_payments_df.printSchema()

root
 |-- event_id: string (nullable = true)
 |-- payment_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- status: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- kafka_ingest_time: timestamp (nullable = true)
 |-- kafka_partition: integer (nullable = true)
 |-- kafka_offset: long (nullable = true)
 |-- bronze_loaded_at: timestamp (nullable = true)
 |-- event_timestamp: timestamp (nullable = true)
 |-- failure_reason: string (nullable = true)
 |-- ingestion_delay_seconds: long (nullable = true)
 |-- is_late: boolean (nullable = true)



In [0]:
# ============================================================
# 13. VERIFY PAYMENT QUARANTINE
# ============================================================

quarantine_payments_df = (
    spark.read
    .format("delta")
    .load(quarantine_path)
)

quarantine_count = quarantine_payments_df.count()

print(f"Payments Quarantine: {quarantine_count}")

display(
    quarantine_payments_df.limit(10)
)

Payments Quarantine: 0


event_id,payment_id,order_id,amount,payment_method,status,timestamp,kafka_ingest_time,kafka_partition,kafka_offset,bronze_loaded_at,event_timestamp,failure_reason,ingestion_delay_seconds,is_late


In [0]:
# ============================================================
# 14. STEP 7 — FULL PAYMENTS RECONCILIATION
# ============================================================

reconciled_total = (
    final_valid_count
    + quarantined_count
    + duplicates_removed_count
)

assert bronze_count == reconciled_total, (
    f"Reconciliation failed: "
    f"bronze={bronze_count}, "
    f"valid+quarantined+duplicates_removed={reconciled_total}"
)

print("====================================================")
print("NEXPULSE — PAYMENTS RECONCILIATION")
print("====================================================")

print(f"Bronze events:          {bronze_count}")
print(f"Valid Silver events:    {final_valid_count}")
print(f"Quarantined events:     {quarantined_count}")
print(f"Duplicates removed:     {duplicates_removed_count}")
print(f"Reconciled total:       {reconciled_total}")

print("----------------------------------------------------")

print(
    f"{final_valid_count} + "
    f"{quarantined_count} + "
    f"{duplicates_removed_count} = "
    f"{reconciled_total}"
)

print("----------------------------------------------------")
print("PASSED: Payments Bronze-to-Silver reconciliation")

NEXPULSE — PAYMENTS RECONCILIATION
Bronze events:          60
Valid Silver events:    59
Quarantined events:     0
Duplicates removed:     1
Reconciled total:       60
----------------------------------------------------
59 + 0 + 1 = 60
----------------------------------------------------
PASSED: Payments Bronze-to-Silver reconciliation


In [0]:
# ============================================================
# 15. FINAL SILVER DATA QUALITY CHECK
# ============================================================

invalid_silver_count = silver_payments_df.filter(
    col("event_id").isNull()
    | col("order_id").isNull()
    | col("payment_id").isNull()
    | col("amount").isNull()
    | (col("amount") < 0)
    | (~col("status").isin(ACCEPTED_STATUSES))
).count()

print(f"Invalid rows in Silver Payments: {invalid_silver_count}")

assert invalid_silver_count == 0

print("PASSED: Silver Payments quality check")

Invalid rows in Silver Payments: 0
PASSED: Silver Payments quality check


In [0]:
try:
    log_pipeline_run(
        pipeline_name="silver_payments",
        source="payments",
        start_time=run_start,
        records_read=bronze_count,
        records_written=final_valid_count,
        records_quarantined=quarantined_count,
        records_deduplicated=duplicates_removed_count,
        status="success"
    )
except Exception as e:
    log_pipeline_run(
        pipeline_name="silver_payments",
        source="payments",
        start_time=run_start,
        status="failed",
        error_message=str(e)[:500]
    )
    raise

[audit] logged run 0d629ba8-10b0-4a34-b61a-84ffaf948112 (success), latency=105.5s
